<a href="https://colab.research.google.com/github/bugulin/SemEval-2026-11/blob/matus-fine-tuning/LLM_fine_tune.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install torch==2.10.0 torchvision==0.25.0 torchaudio==2.10.0 --index-url https://download.pytorch.org/whl/cu128
!pip install -q -U transformers peft accelerate bitsandbytes trl datasets huggingface_hub

We are downloading the needed repositaries and logging to Hugging-face, to be able to download the weihts for the Lamma model - one needs to ask for access from Metha under his token to be granted the exess and download.

In [ ]:
from huggingface_hub import login


# SEM VLOŽ SVOJ HUGGING FACE TOKEN:
login(token="SEM VKLADÁM SVOJ HUGGING FACE TOKEN")

print("Login Successful!")

We are going to download the model if it is not yet in memorry, or we are preparing it.

In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# Verify we are on the GPU
print(f"Working on GPU: {torch.cuda.get_device_name(0)}")

# Load your local file from the sidebar
dataset = load_dataset("json", data_files="/content/my_syllogisms.jsonl", split="train")

# 4. Load Llama 3.1 8B
model_id = "meta-llama/Meta-Llama-3.1-8B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb_config, device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

print(f"Model loaded. Ready to train on {len(dataset)} syllogisms.")

The model is instruction tuned one, it requires special format of the json.

"instruction": entry["syllogism"],

"response": entry["validity"]

Which is not one in sameval, this script changes the formating of the json and return well formated one for the training.

In [ ]:
import json

# Define your file names
input_file = "/content/train_data.json"
output_file = "my_syllogisms.jsonl"

def convert_format(input_path, output_path):
    with open(input_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    # If your JSON is a single list of dictionaries
    if not isinstance(data, list):
        data = [data]

    with open(output_path, 'w', encoding='utf-8') as f:
        for entry in data:
            # Create the new dictionary structure
            new_entry = {
                "instruction": entry["syllogism"],
                "response": str(entry["validity"]).lower()  # Converts False to "false"
            }
            # Write as JSONL (JSON Lines) which is best for training
            f.write(json.dumps(new_entry) + '\n')

    print(f"✅ Conversion complete! {len(data)} rows saved to {output_path}")

# Run the function
convert_format(input_file, output_file)

One more formatting cell.

In [ ]:
def formatting_prompts_func(example):
    # The Trainer now passes a single example (dict) at a time
    text = (
        f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n"
        f"You are a logic expert specializing in syllogisms.<|eot_id|>"
        f"<|start_header_id|>user<|end_header_id|>\n\n"
        f"{example['instruction']}<|eot_id|>"
        f"<|start_header_id|>assistant<|end_header_id|>\n\n"
        f"{example['response']}<|eot_id|>"
    )
    return text # Return a single string

Before reruning LoRa one needs to deatach any LoRa overlay which would be added to the cell.

In [ ]:
if hasattr(model, "unload"):
    model = model.unload()


if hasattr(model, "peft_config"):
    del model.peft_config

print("✅ Model unloaded. You can now re-run your LoraConfig cell safely.")

This is the main training configuration cell, it is the place where all the LoRa parametres one can play with.

In [ ]:
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig


# 1. LoRA Configuration
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules="all-linear",
    bias="none",
    task_type="CAUSAL_LM"
)

training_args = SFTConfig(
    output_dir="./syllogism_model_checkpoints",
    max_length=512,
    dataset_text_field="text",
    packing=False,
    per_device_train_batch_size=3,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=2,
    logging_steps=1,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
)

# 3. Initialize the Trainer
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,
    formatting_func=formatting_prompts_func,
    args=training_args,
)


Time to train, afer everything above is set up and functioning one can trian the LoRa here. It takes some time cca 30min-hour (16GB GPU), depanding on parameters and the dataset...

In [ ]:
trainer.train()

If you run this on google colab do not forget to download your weights.

The following script tests the model´s response on the first query from the dataset.

In [ ]:
test_prompt = dataset[0]['instruction'] # Test with your first example

messages = [
    {"role": "system", "content": "You are a logic expert specializing in syllogisms."},
    {"role": "user", "content": test_prompt},
]
inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt").to("cuda")

# Generate output
outputs = model.generate(**inputs, max_new_tokens=100)
print("\n--- TRAINED MODEL OUTPUT ---")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

In [ ]:
print(dataset)